# 05 - RQ4: Is TabNet's Attention Actually Trustworthy?

**Notebook version:** v17 -- 2026-07-29

- Train TabNet and a control MLP (same depth/width, no attention) on the reduced feature set
- Extract TabNet attention weights; use Integrated Gradients (Sundararajan et al., 2017) for the MLP control
- Compute Spearman correlation of each model's ranking against the SHAP ranking
- Compare TabNet-vs-SHAP agreement to MLP-vs-SHAP agreement to isolate attention's contribution


In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# git pull always runs (cheap, ~seconds) so code is never stale even if Colab's
# 'Restart session' left /content on disk from an earlier session -- only
# pip install (the actual slow part) is skipped via the session marker.
import os
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"
SETUP_MARKER = Path("/content/.secom_setup_done")

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
        SETUP_MARKER.unlink(missing_ok=True)  # fresh clone -- force full setup below

    # Always pull -- cheap, and guarantees code is current even if /content
    # persisted on disk from an earlier session (e.g. Colab 'Restart session'
    # rather than a full 'Disconnect and delete runtime').
    os.chdir(f"/content/{REPO_NAME}")
    !git pull
    os.chdir("/content")

    os.chdir(f"/content/{REPO_NAME}/notebooks")

    already_setup = SETUP_MARKER.exists()
    if not already_setup:
        !pip install -q -r ../requirements.txt
        SETUP_MARKER.touch()
        setup_note = "Ran pip install (git pull always runs regardless)."
    else:
        setup_note = "Skipped pip install -- already done earlier this session. git pull always ran above."

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(setup_note)
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)

import numpy as np
import pandas as pd

from preprocessing import load_raw, screen_missingness, screen_variance, impute_median
from artifacts import load_json
from explainability import per_instance_agreement

X, y = load_raw()
X = impute_median(screen_variance(screen_missingness(X)))

rq3_summary = load_json("rq3_summary")
final_features = rq3_summary["final_features"]
print(f"Using RQ3's reduced feature set ({len(final_features)}): {final_features}")

holdout_split = load_json("rq1_holdout_split")
train_idx, test_idx = holdout_split["train_idx"], holdout_split["test_idx"]
X_train_ho = X.iloc[train_idx][final_features]
X_test_ho = X.iloc[test_idx][final_features]
y_train_ho, y_test_ho = y.iloc[train_idx], y.iloc[test_idx]
print(f"Train: {len(X_train_ho)}, Test (held out): {len(X_test_ho)}")


## Step 1: Train TabNet and a plain MLP control on the same reduced feature set

Both trained on `final_features` only (RQ3's reduced set) so the RQ4
comparison is scoped consistently with the rest of the pipeline. The MLP
control is identical in spirit to TabNet (a feedforward neural net) but
has no attention mechanism -- its role is to isolate how much of any
SHAP-agreement is attributable to attention specifically, vs. just being
a neural network in general.

In [ ]:
import torch
import torch.nn as nn
from pytorch_tabnet.tab_model import TabNetClassifier

X_train_arr = X_train_ho.values.astype("float32")
y_train_arr = y_train_ho.values
X_test_arr = X_test_ho.values.astype("float32")

torch.manual_seed(42)

tabnet = TabNetClassifier(seed=42, verbose=0)
tabnet.fit(X_train_arr, y_train_arr, weights=1, max_epochs=60, patience=10, batch_size=128)
print("TabNet trained.")

class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)

mlp = MLP(len(final_features))
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

# Class-weighted loss, matching class_weight="balanced" used elsewhere in
# this pipeline -- MLP has no built-in imbalance handling otherwise.
pos_weight_ratio = (y_train_arr == 0).sum() / max((y_train_arr == 1).sum(), 1)
sample_weights = np.where(y_train_arr == 1, pos_weight_ratio, 1.0).astype("float32")

X_train_t = torch.tensor(X_train_arr)
y_train_t = torch.tensor(y_train_arr.astype("float32")).unsqueeze(1)
weights_t = torch.tensor(sample_weights).unsqueeze(1)

for epoch in range(200):
    optimizer.zero_grad()
    out = mlp(X_train_t)
    loss = (loss_fn(out, y_train_t) * weights_t).mean()
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"  MLP epoch {epoch}: loss={loss.item():.4f}")
print("MLP trained.")


## Step 2: Extract each model's own explanation

TabNet's `.explain()` is its native attention-derived per-instance feature
importance. Integrated Gradients (Sundararajan, Taly, & Yan, 2017) plays
the same role for the MLP control, since it has no attention layer to
extract -- see docs/synopsis.docx, "Solution to RQ4" for why IG rather
than raw saliency.

In [ ]:
from captum.attr import IntegratedGradients

tabnet_explain, _ = tabnet.explain(X_test_arr)
print(f"TabNet attention explanations: {tabnet_explain.shape}")

ig = IntegratedGradients(mlp)
X_test_t = torch.tensor(X_test_arr, requires_grad=True)
baseline = torch.zeros_like(X_test_t)
ig_attributions = ig.attribute(X_test_t, baseline, target=0).detach().numpy()
print(f"MLP Integrated Gradients: {ig_attributions.shape}")


## Step 3: SHAP explanations for both models

Neither TabNet nor the MLP is a tree model, so `TreeExplainer` doesn't
apply here -- `KernelExplainer` is used instead (model-agnostic, works on
any `predict_proba`-like function), per the synopsis's discussion of
SHAP's approximate mode for non-tree models. `KernelExplainer` is
computationally expensive relative to `TreeExplainer` (no exact/fast path
for neural nets), so this is deliberately run on a subsample of the
held-out set (`N_EXPLAIN`), not all 314 wafers -- increase it if runtime
allows, but expect this cell to be the slowest one in the notebook.

In [ ]:
import shap

N_BACKGROUND = 50
N_EXPLAIN = 40

background = shap.sample(X_train_ho, N_BACKGROUND, random_state=42)
X_explain = X_test_ho.iloc[:N_EXPLAIN]

tabnet_predict_fn = lambda x: tabnet.predict_proba(x.astype("float32"))[:, 1]
explainer_tabnet = shap.KernelExplainer(tabnet_predict_fn, background)
shap_tabnet = np.array(explainer_tabnet.shap_values(X_explain, nsamples=100))
print(f"SHAP (TabNet) computed: {shap_tabnet.shape}")

def mlp_predict_fn(x):
    with torch.no_grad():
        return mlp(torch.tensor(x.astype("float32"))).numpy().flatten()

explainer_mlp = shap.KernelExplainer(mlp_predict_fn, background)
shap_mlp = np.array(explainer_mlp.shap_values(X_explain, nsamples=100))
print(f"SHAP (MLP) computed: {shap_mlp.shape}")


## Step 4: RQ4's actual answer -- does attention agree with SHAP better than IG does?

Per-instance Spearman correlation and top-3 feature-set agreement between
each model's own explanation and its own SHAP values, restricted to the
same `N_EXPLAIN` instances both were computed on.

In [ ]:
tabnet_agreement = per_instance_agreement(shap_tabnet, tabnet_explain[:N_EXPLAIN], top_k=3)
mlp_agreement = per_instance_agreement(shap_mlp, ig_attributions[:N_EXPLAIN], top_k=3)

print("=== TabNet: SHAP vs. attention agreement ===")
print(f"  Mean Spearman correlation: {tabnet_agreement['mean_correlation']:.4f}")
print(f"  Median Spearman correlation: {tabnet_agreement['median_correlation']:.4f}")
print(f"  Top-3 feature-set agreement rate: {tabnet_agreement['top_k_agreement_rate']:.2%}")

print("\n=== MLP control: SHAP vs. Integrated Gradients agreement ===")
print(f"  Mean Spearman correlation: {mlp_agreement['mean_correlation']:.4f}")
print(f"  Median Spearman correlation: {mlp_agreement['median_correlation']:.4f}")
print(f"  Top-3 feature-set agreement rate: {mlp_agreement['top_k_agreement_rate']:.2%}")

print("\n=== RQ4 answer ===")
if tabnet_agreement["mean_correlation"] > mlp_agreement["mean_correlation"]:
    print(
        "TabNet's attention agrees with SHAP more than the MLP control's Integrated "
        "Gradients does -- some support for TabNet's built-in interpretability claim "
        "being more than just 'being a neural network'."
    )
else:
    print(
        "TabNet's attention does NOT agree with SHAP any more than the MLP control's "
        "Integrated Gradients does -- this suggests TabNet's 'built-in interpretability' "
        "claim does not hold up better than a generic post-hoc explanation applied to "
        "any neural net, which is itself a reportable finding, not a null result."
)


## Next steps

- Report both agreement rates and the RQ4 conclusion in the capstone
  write-up
- Note the SHAP sample size limitation (`N_EXPLAIN` wafers, not the full
  314) explicitly if reporting these numbers -- a smaller, faster proxy
  for the full held-out set, not the full set itself

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "05_attention_comparison_rq4"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"
live_export_available = False

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed. This only
    # works when there's an actual live frontend attached (i.e. you're running
    # this cell interactively yourself) -- it returns None instead of raising
    # when run unattended (e.g. via 00_run_all.ipynb's automated execution),
    # so that case is caught explicitly here rather than left to crash with a
    # raw TypeError, which used to make 00_run_all's --allow-errors flag mask
    # *real* failures elsewhere in the notebook, not just this expected one.
    from google.colab import _message
    response = _message.blocking_request('get_ipynb', timeout_sec=30)
    if response is not None:
        ipynb_content = response['ipynb']
        with open(export_path, 'w') as f:
            json.dump(ipynb_content, f)
        live_export_available = True
    else:
        print(
            "No live Colab frontend detected (expected when run via "
            "00_run_all.ipynb) -- skipping the live export. 00_run_all does its "
            "own separate HTML export against the already-executed file instead."
        )

if live_export_available or not IN_COLAB:
    html_output = f"{NOTEBOOK_NAME}.html"
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print(f"Exported to {html_output}")

    if IN_COLAB and result.returncode == 0:
        from google.colab import files
        files.download(html_output)
